In [29]:
import os
import requests
import mysql.connector
import pandas as pd
from dotenv import load_dotenv

# Cargar las variables de entorno del archivo .env
load_dotenv("/home/jovyan/work/.env")

def cargar_system_prompt():
    ruta_prompt = '/home/jovyan/work/notebooks/prompt_sistema.md'
    try: 
        with open(ruta_prompt, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        print("Error: no se encontró el archivo de prompt de sistema.")
        return ""

def text_to_sql(pregunta_usuario):
    url = os.getenv("OLLAMA_URL", "http://ollama:11434/api/generate")
    model = os.getenv("LLM_MODEL", "llama3.2")
    
    system_prompt = cargar_system_prompt()
    if not system_prompt:
        raise Exception("No se pudo cargar el prompt del sistema.")

    full_prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nTraduce esta pregunta a SQL: {pregunta_usuario}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
    
    payload = {
        "model": model,
        "prompt": full_prompt,
        "stream": False,
        "options": {
            "temperature": 0.0 # Creatividad cero para aumentar precisión
        }
    }
    
    response = requests.post(url, json=payload)
    resultado = response.json()
    
    sql_generado = resultado['response'].strip()
    return sql_generado

def ejecutar_consulta(sql):
    conn = mysql.connector.connect(
        host=os.getenv("DB_HOST"),
        port=int(os.getenv("DB_PORT")),
        database=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD")
    )
    try:
        with conn:
            df = pd.read_sql(sql, conn)
        return df
    except Exception as e:
        print(f"Error sintáctico en la base de datos: {e}")
        raise e


def preguntar_al_agente(pregunta):
    print(f"Pregunta del usuario: '{pregunta}'")
    try:
        sql = text_to_sql(pregunta)
        print(f"SQL Generado por el modelo:\n{sql}\n")
        
        print("Conectando a Aiven...")
        df_resultado = ejecutar_consulta(sql)

        print("Resultado de la Base de Datos:")
        display(df_resultado)
        
    except Exception as e:
        print(f"Ocurrió un error: {e}")

In [31]:
print("Prueba rápida de IA")
print("-" * 60)

# preguntar_al_agente("cuál es el libro más prestado?")
preguntar_al_agente("qué hora es?")

Prueba rápida de IA
------------------------------------------------------------
Pregunta del usuario: 'qué hora es?'
SQL Generado por el modelo:
SELECT NOW()

Conectando a Aiven...


/tmp/ipykernel_8907/3313579556.py:54: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


Resultado de la Base de Datos:


,NOW()
0,2026-06-17 15:42:41


In [7]:
print("¡Agente BiblioIA Activado! Escribí 'salir' para terminar.")
print("-" * 60)

while True:
    pregunta = input("\nIngresá tu pregunta para la biblioteca: ")
    if pregunta.lower() in ['salir', 'chau','adios','adiós','exit', 'quit']:
        print("Chau")
        break
    if pregunta.strip() == "":
        continue
        
    # Llama a tu función principal
    preguntar_al_agente(pregunta)

¡Agente BiblioIA Activado! Escribí 'salir' para terminar.
------------------------------------------------------------
Pregunta del usuario: 'qué libros se prestaron en los últimos dos años?'
SQL Generado por el modelo:
SELECT libro.titulo 
FROM prestamo p
JOIN libro l ON p.isbn = l.isbn
WHERE p.fecha_prestamo >= DATE_SUB(CURDATE(), INTERVAL 2 YEAR) AND p.fecha_devolucion IS NULL;

Conectando a Aiven...


/tmp/ipykernel_8907/1538466198.py:87: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


Error sintáctico en la base de datos: Execution failed on sql 'SELECT libro.titulo 
FROM prestamo p
JOIN libro l ON p.isbn = l.isbn
WHERE p.fecha_prestamo >= DATE_SUB(CURDATE(), INTERVAL 2 YEAR) AND p.fecha_devolucion IS NULL;': 1054 (42S22): Unknown column 'libro.titulo' in 'field list'
Ocurrió un error: Execution failed on sql 'SELECT libro.titulo 
FROM prestamo p
JOIN libro l ON p.isbn = l.isbn
WHERE p.fecha_prestamo >= DATE_SUB(CURDATE(), INTERVAL 2 YEAR) AND p.fecha_devolucion IS NULL;': 1054 (42S22): Unknown column 'libro.titulo' in 'field list'
Chau


In [ ]:
import requests

try:
    # Cambiá 'llama3.2' por el modelo exacto que bajaron si usaron otro
    res = requests.post("http://ollama:11434/api/generate", 
                        json={"model": "llama3.2", "prompt": "Hola, estás vivo?", "stream": False}, 
                        timeout=60)
    print("Respuesta de Ollama:", res.json()['response'])
except Exception as e:
    print("Error al conectar con Ollama:", e)